# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application. It uses **Gemma 4**, Google's open-weight model, running inside this notebook. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight, both running in this notebook):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings.
- `gemma-4-E2B-it` (Google) writes the answers.

### Before you start: select a GPU
This notebook runs the Gemma model inside Google Colab and needs a GPU. Select **Runtime → Change runtime type → T4 GPU → Save**. The free T4 GPU is sufficient.

No API key is required. Gemma 4 is released under the Apache 2.0 licence and downloads without a Hugging Face account.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top. The installation cell does not need to be repeated after a restart unless the runtime was disconnected.

## Step 0: Installation (run once)

In [ ]:
%%capture
# Installs Unsloth and compatible library versions (takes 2-4 minutes; output is hidden).
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
!pip install pypdf sentence-transformers "gradio>=6,<7"

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU found. Select Runtime → Change runtime type → T4 GPU, then run the notebook again from the top.")

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [ ]:
# STUDENT TODO: the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and longer prompts
CHUNK_SIZE = 120

# STUDENT TODO: the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO: the number of chunks given to Gemma for each question.
TOP_K = 3

# STUDENT TODO: the rules Gemma follows when answering. The rule to use only the context prevents invented answers.
ANSWER_RULES = """You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences."""

TEMPERATURE = 0.0       # 0.0 keeps answers consistent and close to the document
MAX_NEW_TOKENS = 256    # the maximum length of each answer
MAX_SEQ_LENGTH = 4096   # the maximum prompt length the model accepts (question + retrieved chunks)

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

## Step 2: Load the models
Downloads Gemma 4 E2B (about 4 GB in 4-bit form) and the embedding model (about 90 MB). This takes 2–3 minutes on the first run.

In [ ]:
from unsloth import FastModel

# STUDENT TODO: the Gemma model to use.
#   "unsloth/gemma-4-E2B-it"  Gemma 4 E2B (about 2B effective parameters). Fits comfortably on a free T4. (default)
#   "unsloth/gemma-4-E4B-it"  Gemma 4 E4B (about 4B effective parameters). More capable, slower, uses more memory.
MODEL_NAME = "unsloth/gemma-4-E2B-it"

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None,              # selected automatically
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,       # 4-bit quantisation reduces memory use
    full_finetuning = False,
)
print(f"Loaded {MODEL_NAME}")

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

In [ ]:
def generate(user_text, system_text=None, max_new_tokens=256, temperature=0.0):
    """Sends one message to Gemma and returns its reply as text.
    temperature = 0.0 gives the same answer every time; higher values give more varied answers."""
    messages = []
    if system_text:
        messages.append({"role": "system", "content": [{"type": "text", "text": system_text}]})
    messages.append({"role": "user", "content": [{"type": "text", "text": user_text}]})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")

    sampling = {"do_sample": True, "temperature": temperature, "top_p": 0.95, "top_k": 64} if temperature > 0 else {"do_sample": False}
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, **sampling)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Quick test: Gemma on its own, without any document.
print(generate("In one sentence, what is an open-weight AI model?", max_new_tokens=80))

## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [ ]:
# STUDENT TODO: replace with text relevant to your use case, or upload a PDF in the application instead.
SAMPLE_PAGES = [
    """CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.""",

    """Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. Final exam: 25%.
    Class participation: 5%. A score of 90% or above earns an A, 80 to 89% a B, 70 to 79% a C, and 60 to 69% a D.
    Late submissions lose 10% per day, up to a maximum of 3 days, after which they receive zero. Each student has
    two grace days per semester that may be used to submit any lab late without penalty.""",

    """Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions.
    Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling.
    Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to
    3 students. Week 14: review. The final exam is a 2-hour written exam during the examination period.""",

    """Policies. Attendance: students must attend at least 75% of lectures to be eligible for the final exam.
    Academic integrity: students may discuss ideas with classmates, but all submitted code must be their own. AI
    tools may be used for learning and debugging, but their use must be stated in a comment at the top of the file.
    Laptops: bring a laptop to every lab. Students without a laptop may reserve a machine in the Computer Lab.
    Textbook: "Automate the Boring Stuff with Python" (available free online).""",
]
print(f"Sample document contains {len(SAMPLE_PAGES)} pages.")

## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [ ]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    reply = generate(prompt, system_text=ANSWER_RULES, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE)
    return reply, found


# Quick test with the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [ ]:
import os
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running

APP_TITLE = "Document Question Answering"  # STUDENT TODO: rename for your use case
APP_DESCRIPTION = "Upload a PDF and ask questions. Answers are based only on the document and include page references."


def on_upload(file):
    if file is None:
        return "No file uploaded."
    try:
        return build_knowledge_base(read_pdf(file), os.path.basename(file))
    except Exception as e:
        raise gr.Error(f"The PDF could not be read: {e}")


def on_ask(question):
    if not question.strip():
        raise gr.Error("Enter a question first.")
    reply, found = answer(question)
    sources = "\n\n".join(f"**Page {c['page']}** (similarity {c['score']:.2f})\n> {c['text']}" for c in found)
    return reply, sources


with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        pdf_input = gr.File(label="Upload a PDF", file_types=[".pdf"], type="filepath")
        with gr.Column():
            status_md = gr.Markdown(f"Current document: **{knowledge_base['name']}**")
            sample_btn = gr.Button("Use the sample document")
    question_box = gr.Textbox(label="Question", placeholder="For example: How is the course graded?")
    ask_btn = gr.Button("Ask", variant="primary")
    gr.Examples(["How is the course graded?", "When are office hours?", "Can I use AI tools?",
                 "Who won the 2022 World Cup?"], inputs=question_box)  # STUDENT TODO: questions for your document
    answer_md = gr.Markdown()
    with gr.Accordion("Retrieved passages (the text given to Gemma)", open=False):
        sources_md = gr.Markdown()

    pdf_input.upload(on_upload, inputs=pdf_input, outputs=status_md)
    sample_btn.click(lambda: build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"), outputs=status_md)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, sources_md])

demo.launch(share=True)

## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| "No GPU found" | Select **Runtime → Change runtime type → T4 GPU**, then run all cells from the top. |
| `CUDA out of memory` | Select **Runtime → Restart session** and run the cells again. Reduce `MAX_SEQ_LENGTH` or `max_new_tokens`, or use the E2B model. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| The first answer is slow | Expected. The model is compiled on first use. Later answers are faster. |
| Installation errors | Select **Runtime → Disconnect and delete runtime**, then run the notebook again from the top. |
| GPU usage limit reached | Colab limits free GPU time per account. Continue with another team member's Google account. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |